In [15]:
import pytrendline
import pandas as pd
import os
import time
import akshare as ak
import warnings
warnings.filterwarnings('ignore')
# 1. Construct candlestick data. This example just grabs data from a fixture
candles_df = ak.stock_zh_a_hist(symbol="600089", period="daily", start_date="20230101", end_date='20240607', adjust="qfq")
candles_df['Date'] = pd.to_datetime(candles_df['日期'])


candlestick_data = pytrendline.CandlestickData(
  df=candles_df,
  time_interval="1d", # choose between 1m,3m,5m,10m,15m,30m,1h,1d
  open_col="开盘", # name of the column containing candle "Open" price
  high_col="最高", # name of the column containing candle "High" price
  low_col="最低", # name of the column containing candle "Low" price
  close_col="收盘", # name of the column containing candle "Close" price
  datetime_col="Date" # name of the column containing candle datetime price (use none if datetime is in index)
)

print("📈 📉 Starting call to pytrendline.detect ... (this could take a while on a large candlestick dataset)")

detect_start_time = time.time()

# 2. Find trendlines. Results are returned in the form of
#     a. A pandas dataframe table containing trendline found per row
#     b. A pandas series containing pivot points
results = pytrendline.detect(
  candlestick_data=candlestick_data,

  # Choose between BOTH, SUPPORT or RESISTANCE
  trend_type=pytrendline.TrendlineTypes.BOTH,
  # Specify if you require the first point of a trendline to be a pivot
  first_pt_must_be_pivot=False,
  # Specify if you require the last point of the trendline to be a pivot
  last_pt_must_be_pivot=False,
  # Specify if you require all trendline points to be pivots
  all_pts_must_be_pivots=False,
  # Specify if you require one of the trendline points to be global max or min price
  trendline_must_include_global_maxmin_pt=False,
  # Specify minimum amount of points required for trendline detection (NOTE: must be at least two)
  min_points_required=3,
  # Specify if you want to ignore prices before some date
  scan_from_date=None,
  # Specify if you want to ignore 'breakout' lines. That is, lines that interesect a candle
  ignore_breakouts=True,
  # Specify and override to default config (See docs on how)
  config={    
 # By default set to allow all angles but min or max can be set to 0 to only allow possitive / negative slopes
  'max_allowable_support_slope': lambda candles: 0.02,
  'min_allowable_support_slope': lambda candles: -0.02,
  'max_allowable_resistance_slope': lambda candles: 0.02,
  'min_allowable_resistance_slope': lambda candles: -0.02,
  'max_allowable_support_last_price': lambda candles: 18,
  'min_allowable_support_last_price': lambda candles: 12,
  'max_allowable_resistance_last_price': lambda candles: 18,
  'min_allowable_resistance_last_price': lambda candles: 12,

  }
)

detect_end_time = time.time()

print("✅ pytrendline.detect took {:.4f}s".format(detect_end_time - detect_start_time))

# 3. Plot the trendlines found
outf = pytrendline.plot(
  results=results,
  filedir='.',
  filename='example_output.html',
)

print("💾 Trendline results saved in {}".format(outf))
os.system("open " + outf)

📈 📉 Starting call to pytrendline.detect ... (this could take a while on a large candlestick dataset)
✅ pytrendline.detect took 111.4410s
💾 Trendline results saved in ./example_output.html


1

In [25]:
df = results['support_trendlines']

df#[df['overall_rank'] >= 1]

,id,trendtype,pointset_indeces,pointset_dates,starts_at_index,starts_at_date,ends_at_index,ends_at_date,is_breakout,breakout_index,...,slope,price_at_last_date,score,includes_global_max_or_min,global_maxs_or_mins,price_at_next_future_date,duplicate_group_id,is_best_from_duplicate_group,overall_rank,rank_within_group
2,"S-[3,232,235]",SUPPORT,"[3, 232, 235]","[2023-01-06 00:00:00, 2023-12-18 00:00:00, 202...",3,2023-01-06,235,2023-12-21,False,None,...,-0.002200,12.037249,1606.833027,True,"[232, 235]",12.030349,2000,True,1,1
4,"S-[236,266,317]",SUPPORT,"[236, 266, 317]","[2023-12-22 00:00:00, 2024-02-05 00:00:00, 202...",236,2023-12-22,317,2024-04-26,False,None,...,0.002126,13.710000,1494.565217,False,"[232, 235]",13.716667,2003,True,2,1
1,"S-[2,232,235]",SUPPORT,"[2, 232, 235]","[2023-01-05 00:00:00, 2023-12-18 00:00:00, 202...",2,2023-01-05,235,2023-12-21,False,None,...,-0.002052,12.089304,1397.357724,True,"[232, 235]",12.082870,2000,False,None,2
3,"S-[235,316,317]",SUPPORT,"[235, 316, 317]","[2023-12-21 00:00:00, 2024-04-25 00:00:00, 202...",235,2023-12-21,317,2024-04-26,False,None,...,0.002834,13.748889,1345.108696,True,"[232, 235]",13.757778,2003,False,None,2
0,"S-[1,232,235]",SUPPORT,"[1, 232, 235]","[2023-01-04 00:00:00, 2023-12-18 00:00:00, 202...",1,2023-01-04,235,2023-12-21,False,None,...,-0.001946,12.126364,1278.683575,True,"[232, 235]",12.120260,2000,False,None,3
7,"S-[266,316,317]",SUPPORT,"[266, 316, 317]","[2024-02-05 00:00:00, 2024-04-25 00:00:00, 202...",266,2024-02-05,317,2024-04-26,False,None,...,0.001977,13.673600,1083.018273,False,"[232, 235]",13.679800,2003,False,None,3
6,"S-[237,239,266]",SUPPORT,"[237, 239, 266]","[2023-12-25 00:00:00, 2023-12-27 00:00:00, 202...",237,2023-12-25,266,2024-02-05,False,None,...,-0.000330,13.109310,220.012139,False,"[232, 235]",13.108276,2006,True,3,1
5,"S-[236,266,316,317]",SUPPORT,"[236, 266, 316, 317]","[2023-12-22 00:00:00, 2024-02-05 00:00:00, 202...",236,2023-12-22,317,2024-04-26,False,None,...,0.002033,13.678500,77.554140,False,"[232, 235]",13.684875,2003,False,None,4
10,"S-[325,329,344]",SUPPORT,"[325, 329, 344]","[2024-05-13 00:00:00, 2024-05-17 00:00:00, 202...",325,2024-05-13,344,2024-06-07,False,None,...,-0.004363,14.170000,36.878882,False,"[232, 235]",14.156316,2009,True,5,1
9,"S-[316,317,344]",SUPPORT,"[316, 317, 344]","[2024-04-25 00:00:00, 2024-04-26 00:00:00, 202...",316,2024-04-25,344,2024-06-07,False,None,...,0.007629,14.170000,11.531503,False,"[232, 235]",14.193929,2009,False,None,2
